In [ ]:
# -----------------------------------------------------------------------------
# Block 0 — imports
# -----------------------------------------------------------------------------
from pathlib import Path

import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt

import os
import numpy as np

os.getcwd()
# -----------------------------------------------------------------------------

In [ ]:
# Block 1 — where the data lives (set ROOT once for the secure environment)
# -----------------------------------------------------------------------------
# Point ROOT at the folder that holds LISS/, EXPOSOME/, MISC/.
ROOT = Path("../../../source")

## EXPOSOME Data
EXPOSOME = ROOT / "EXPOSOME"

livability = Path("../../data/livability")
liv_files_score_PC4_2024 = livability / "Leefbarometer-scores PC4 2022-2024.csv" 
urban_file = ROOT / "EXPOSOME" / "Urban" / "UR_B15_XX_XX_15_v3.tif"
noise_file = ROOT / "EXPOSOME" / "Noise" / "

# LISS = ROOT / "LISS"

# AVARS_CSV    = LISS / "BackgroudVariables/avars_202401_EN_1.0p/avars_202401_EN_1.0p.csv"
# POL18_CSV    = LISS / "LISSCoreStudyWave12to218/PoliticsAndValues/wave18/cv26r_EN_1.0p.csv"
# PC6_PANEL_CSV = LISS / "LISS_PC6_2020-2026/PC6_LISS_2020-2026.csv"

# PC6_POLY_GPKG = ROOT / "MISC/2025-cbs_pc6_2024_v1/cbs_pc6_2024_v1.gpkg"
# NO2_PATH      = ROOT / "EXPOSOME" / "AirPollution" / "NO2B25_AAV_XX_XX_05_v2.tif"

# Wave 18 of PoliticsAndValues was fielded in 2022H2 -> use the july2022 postcode.
# PC6_COLUMN = "july2022"

# Engine for all geopandas reads (fiona is broken on the secure machine).
GEO_ENGINE = "pyogrio"

In [ ]:
## 1) Loading in PC6 and it's geometry data
## The PC6 panel is a normal comma-separated CSV, one column per half-year.
# pc6_panel = pd.read_csv(PC6_PANEL_CSV)
# print(f"loaded {PC6_PANEL_CSV.name}: {pc6_panel.shape[0]:,} rows, {pc6_panel.shape[1]} cols")

# PC6_PANEL_CSV = LISS / "LISS_PC6_2020-2026/PC6_LISS_2020-2026.csv"

PC6_POLY_GPKG = ROOT / "MISC/2025-cbs_pc6_2024_v1/cbs_pc6_2024_v1.gpkg"
# NO2_PATH      = ROOT / "EXPOSOME" / "AirPollution" / "NO2B25_AAV_XX_XX_05_v2.tif"

## define column month / year relevant for livability (july 2024, half way through the year)
# PC6_COLUMN = "july2024"

# Engine for all geopandas reads (fiona is broken on the secure machine).
GEO_ENGINE = "pyogrio"

## norming PC6 column to be sure it has correct format
def norm_pc6(s):
    # "1509 BM" -> "1509BM" so the LISS key matches the CBS postcode6 format.
    return s.astype(str).str.replace(" ", "", regex=False).str.upper()

# # PC6 polygons (CBS): load, normalise the postcode column, ensure RD (EPSG:28992).
pc6_poly = gpd.read_file(PC6_POLY_GPKG, engine=GEO_ENGINE)
print("pc6_poly cols:", pc6_poly.columns.tolist()[:6], "...")

for cand in ["pc6", "postcode6", "PC6", "postcode", "Postcode6"]:
    if cand in pc6_poly.columns:
        pc6_poly = pc6_poly.rename(columns={cand: "pc6"})
        break
else:
    raise KeyError(f"No postcode column found in {pc6_poly.columns.tolist()}")

pc6_poly["pc6"] = norm_pc6(pc6_poly["pc6"])

## correct geocoding
if pc6_poly.crs is None or pc6_poly.crs.to_epsg() != 28992:
    pc6_poly = pc6_poly.to_crs(28992)
print("pc6_poly:", len(pc6_poly), "| CRS:", pc6_poly.crs)

## loading in exposome data, here, data from urban file (.tif file) 
## confirm it opens and report its grid.
with rasterio.open(urban_file) as src:
    raster_crs = src.crs
    data_exposure = src.read(1)
    transform = src.transform
    print(f"urban raster — CRS: {src.crs}, res: {src.res}, dtype: {src.dtypes[0]}")

In [ ]:
## link 
## 1): First take a centroid on PC6 level
pc6_poly = gpd.GeoDataFrame(pc6_poly, geometry="geometry", crs=28992)

## count areas without PC6
n_unmatched = pc6_poly.geometry.isna().sum()
print(f"Unmatched PC6: {n_unmatched} / {len(pc6_poly)} ({n_unmatched / len(pc6_poly):.1%})")
pc6_poly = pc6_poly.dropna(subset=["geometry"]).copy()

## create centroid column
pc6_poly["centroid"] = pc6_poly.geometry.centroid

## sample exposome varible (here urban) at each centroid. The raster is EPSG:3035, so reproject the
## centroids into the raster CRS first, then read the values.
cent_raster = gpd.GeoSeries(pc6_poly["centroid"], crs=28992).to_crs(raster_crs)
coords = [(p.x, p.y) for p in cent_raster]

with rasterio.open(urban_file) as src:
    nodata = src.nodata
    pc6_poly["urban"] = [v[0] for v in src.sample(coords)]


# Drop nodata / off-grid samples so they don't poison the colour scale.
if nodata is not None:
    pc6_poly.loc[pc6_poly["urban"] == nodata, "urban"] = pd.NA
pc6_poly["urban"] = pd.to_numeric(pc6_poly["urban"], errors="coerce")
pc6_poly = pc6_poly.dropna(subset=["urban"]).copy()

# print(f"Respondents with a valid NO2 value: {len(liss):,}")
# print(liss[["nomem_encr", "pc6", "no2"]].head())

print(f"PC6 with a valid 'urban' value: {len(pc6_poly):,}")

In [ ]:
## break it down to PC4 (format to merge with the livability scores)
## break it down to PC4
pc6_poly["pc4"] = pc6_poly["pc6"].str[:4]
pc4_urban = pc6_poly.groupby("pc4")["urban"].mean().reset_index()
print(f"PC4 areas with urban values: {len(pc4_urban)}")

display(pc4_urban.head())

In [ ]:
## Optional: Creating Maps (color coded)
